In [18]:
import requests
import os
import json
import pandas as pd
from dotenv import load_dotenv, dotenv_values
from requests.exceptions import HTTPError, ConnectionError, Timeout, RequestException
from pathlib import Path

In [19]:
load_dotenv(Path.cwd().parent / ".env")

False

In [20]:
URL_BASE = "https://api.company-information.service.gov.uk"
URL_META_DATA = "https://document-api.company-information.service.gov.uk/document"
TIMEOUT = 10  # seconds

In [21]:
#Company House

def get(endpoint, params=None, headers=None):
    url = f"{URL_BASE}/{endpoint}"
    response = requests.get(
        url,
        auth=(os.getenv("CH_API"),""),
        headers=headers,
        params=params,
        timeout=TIMEOUT
        )
    response.raise_for_status()
    print(response.status_code)
    return response.json()

def get_meta_data(endpoint, params=None, headers=None):
    url = f"{URL_META_DATA}/{endpoint}"
    response = requests.get(
        url,
        auth=(os.getenv("CH_API"),""),
        headers=headers,
        params=params,
        timeout=TIMEOUT
        )
    response.raise_for_status()
    return response

In [22]:
import time, threading, itertools, re
from concurrent.futures import ThreadPoolExecutor, as_completed
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k): return x            # no-op fallback if tqdm isn't installed

# --- Multi-key throttled Companies House client (uses all API keys -> ~Nx faster)
# CH's 600-requests/5-min limit is PER API KEY. We load every CH_API* key, give
# each its own Session + pacer (~1.7 req/s), and spread requests across them via a
# thread pool. Aggregate throughput ~= N_keys * 1.7 req/s (4 keys ~= 6.7 req/s).

def _clean_key(v):
    """Extract a key value written as  KEY = 'abc',  /  "abc",  /  abc  (the .env
       here stores them Python-list style with quotes and trailing commas)."""
    v = v.strip()
    m = re.search(r"""['"]([^'"]+)['"]""", v)   # value inside the first quote pair
    return m.group(1) if m else v.rstrip(",").strip()


def _load_ch_keys(names=("CH_API", "CH_API_2", "CH_API_3", "CH_API_4")):
    """Load keys tolerantly from the repo-root .env (handles spaces around '=',
       quotes and trailing commas that python-dotenv rejects), then env vars."""
    parsed = {}
    for cand in (Path.cwd().parent.parent / ".env",
                 Path.cwd().parent / ".env",
                 Path.cwd() / ".env"):
        if cand.exists():
            for line in cand.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                parsed[k.strip()] = _clean_key(v)
            break
    keys = [parsed.get(n) or os.getenv(n) for n in names]
    return [k for k in keys if k]


_KEYS = _load_ch_keys()
if not _KEYS:
    raise RuntimeError("No Companies House API keys found (checked .env and env vars)")

_MIN_INTERVAL = 0.6           # seconds between calls PER KEY (~1.7/s; CH cap 2.0/s each)
MAX_WORKERS   = 2 * len(_KEYS)
print(f"{len(_KEYS)} API key(s) loaded -> up to ~{len(_KEYS) / _MIN_INTERVAL:.1f} req/s "
      f"with {MAX_WORKERS} workers")


class _Lane:
    """One API key: dedicated Session + a lock enforcing min-interval pacing."""
    __slots__ = ("session", "lock", "last")
    def __init__(self, key):
        self.session = requests.Session(); self.session.auth = (key, "")
        self.lock = threading.Lock(); self.last = 0.0
    def pace(self):
        with self.lock:
            wait = _MIN_INTERVAL - (time.monotonic() - self.last)
            if wait > 0:
                time.sleep(wait)
            self.last = time.monotonic()


_LANES = [_Lane(k) for k in _KEYS]
_rr = itertools.count(); _rr_lock = threading.Lock()

def _next_lane():
    with _rr_lock:
        i = next(_rr)
    return _LANES[i % len(_LANES)]


def ch_get(path, params=None, max_retries=5):
    """Thread-safe, key-rotating, throttled GET. Returns a Response (200/404/etc.
       for the caller to interpret) or None if it gave up. Handles 429 (waits out
       the window) and transient network/5xx errors with exponential backoff."""
    url = f"{URL_BASE}{path}"
    for attempt in range(max_retries):
        lane = _next_lane()
        lane.pace()
        try:
            r = lane.session.get(url, params=params, timeout=TIMEOUT)
        except (ConnectionError, Timeout):
            time.sleep(2 ** attempt); continue
        if r.status_code == 429:                       # rare with per-key pacing
            time.sleep(int(r.headers.get("Retry-After", 60)) + 1); continue
        if r.status_code >= 500:                        # transient server error
            time.sleep(2 ** attempt); continue
        return r
    return None


def parallel_fill(df, todo_index, fn, cols, out_path,
                  desc="fetch", max_workers=None, checkpoint_every=400):
    """Concurrently apply fn(com_num) over todo_index rows and write results into
       `cols` (str, or list matching fn's tuple return). Only network I/O runs in
       threads; all DataFrame writes + checkpoints happen on the main thread, so no
       df locking is needed. Resumable + checkpointed + live rate/ETA."""
    cols = [cols] if isinstance(cols, str) else list(cols)
    todo_index = list(todo_index)
    n_todo = len(todo_index)
    workers = max_workers or MAX_WORKERS
    print(f"{n_todo} to fetch (of {len(df)}) — {len(_LANES)} keys x {workers} workers")
    t0 = time.time(); done = 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        fut2idx = {ex.submit(fn, df.at[idx, "com_num"]): idx for idx in todo_index}
        for fut in tqdm(as_completed(fut2idx), total=n_todo, desc=desc):
            idx = fut2idx[fut]
            try:
                res = fut.result()
            except Exception:
                res = "error" if len(cols) == 1 else tuple("error" for _ in cols)
            if len(cols) == 1:
                df.at[idx, cols[0]] = res
            else:
                for c, v in zip(cols, res):
                    df.at[idx, c] = v
            done += 1
            if done % checkpoint_every == 0:
                df.to_csv(out_path, index=False)
                rate = done / (time.time() - t0)
                eta = (n_todo - done) / rate / 60 if rate else 0
                print(f"  {done}/{n_todo} | {rate:.2f}/s | ETA ~{eta:.0f} min | checkpoint saved")
    df.to_csv(out_path, index=False)
    print(f"Saved -> {out_path}")

4 API key(s) loaded -> up to ~6.7 req/s with 8 workers


## Bulk: add `account_type` — new companies only

Scans **every** CSV in `company_data/company_raw/`, dedupes on `com_num`, and pulls `accounts.last_accounts.type` **only for company numbers not already processed**. Drop new raw files into `company_raw/` and re-run — it fetches just the new ones and appends them (old companies are never re-pulled).

Output → `company_data/company_with_acctype/com_names_with_acctype.csv`

In [ ]:
# --- Bulk pull: account_type for NEW companies only (multi-key concurrent) ------
# Scans every CSV in company_raw/, dedupes on com_num, fetches account_type ONLY
# for company numbers not already in the output, and appends them.
BASE        = Path("company_data")
RAW_DIR     = BASE / "company_raw"
ACCTYPE_OUT = BASE / "company_with_acctype" / "com_names_with_acctype.csv"

# 1. Combine every raw file; one row per company (raw files repeat com_num per SIC).
raw = pd.concat([pd.read_csv(f, dtype=str) for f in sorted(RAW_DIR.glob("*.csv"))],
                ignore_index=True).drop_duplicates("com_num")

# 2. Load prior results; NEW companies = those not already processed.
if ACCTYPE_OUT.exists():
    done = pd.read_csv(ACCTYPE_OUT, dtype=str)
else:
    done = pd.DataFrame(columns=list(raw.columns) + ["account_type"])

new = raw[~raw["com_num"].isin(done["com_num"])].copy()
new["account_type"] = pd.NA
df = pd.concat([done, new], ignore_index=True)   # keep old results, append new rows


def fetch_account_type(company_num):
    """Last-accounts type, or a sentinel: 'no_accounts', 'not_found', 'error'."""
    r = ch_get(f"/company/{str(company_num).strip()}")
    if r is None:              return "error"
    if r.status_code == 404:   return "not_found"
    if r.status_code != 200:   return "error"
    t = (r.json().get("accounts", {}).get("last_accounts", {}).get("type"))
    return t or "no_accounts"


todo = df.index[df["account_type"].isna()]
print(f"{len(new)} new companies from raw | {len(done)} already done")
parallel_fill(df, todo, fetch_account_type, "account_type", ACCTYPE_OUT, desc="account_type")
df["account_type"].value_counts(dropna=False)


In [7]:
# --- Remove perfect duplicate rows from the acctype file -----------------------
# A "perfect duplicate" = every column identical. The pull already dedupes on
# com_num, so this is now a harmless safety net (expect 0 removed).
dedup = pd.read_csv(ACCTYPE_OUT, dtype=str)
before = len(dedup)
dedup = dedup.drop_duplicates()          # all columns must match
removed = before - len(dedup)

dedup.to_csv(ACCTYPE_OUT, index=False)
print(f"Removed {removed} perfect duplicate row(s): {before} -> {len(dedup)} rows in {ACCTYPE_OUT}")


Removed 0 perfect duplicate row(s): 25172 -> 25172 rows in company_data/company_with_acctype/com_names_with_acctype.csv


## Filter to SME account types

Keep only companies whose `account_type` is one of `small`, `micro-entity`, `total-exemption-full`, `total-exemption-small`, `medium`. Output → `com_names_sme.csv`.

In [8]:
# --- Keep only SME-relevant account types --------------------------------------
# Rebuilds the SME set from the (now larger) acctype file, and CARRIES FORWARD any
# lloyds_customer labels already computed -- so new SME companies get NA and only
# they are checked in the next cell.
SME_TYPES = ["small", "micro-entity", "total-exemption-full",
             "total-exemption-small", "medium"]

BASE    = Path("company_data")
ACCTYPE_OUT = BASE / "company_with_acctype" / "com_names_with_acctype.csv"
SME_OUT     = BASE / "company_sme_with_acctype" / "com_names_sme.csv"

full = pd.read_csv(ACCTYPE_OUT, dtype=str)
sme  = full[full["account_type"].isin(SME_TYPES)].copy()

if SME_OUT.exists():
    prev = pd.read_csv(SME_OUT, dtype=str)[["com_num", "lloyds_customer"]]
    sme = sme.merge(prev, on="com_num", how="left")   # keep existing True/False labels
else:
    sme["lloyds_customer"] = pd.NA

sme.to_csv(SME_OUT, index=False)
print(f"SME companies: {len(sme)} of {len(full)} -> {SME_OUT}")
print(f"  Lloyds check pending for {sme['lloyds_customer'].isna().sum()} new SME companies")
sme["account_type"].value_counts()


SME companies: 13544 of 25172 -> company_data/company_sme_with_acctype/com_names_sme.csv
  Lloyds check pending for 9196 new SME companies


account_type
micro-entity             8751
total-exemption-full     4567
small                     160
total-exemption-small      58
medium                      8
Name: count, dtype: int64

## Flag Lloyds borrowers

For each SME company, pull its charges (all pages) and check whether any charge was ever held by a Lloyds Bank entity. Adds a `lloyds_customer` (True/False) column to `com_names_sme.csv`.

Note: matches "Lloyds" (bank) but **not** "Lloyd's" of London (insurance market). Other Lloyds Banking Group brands (Bank of Scotland, Halifax, Scottish Widows) are excluded by default — add them to `LLOYDS_PATTERNS` to include the whole group.

In [ ]:
import re

# --- Flag SME companies that ever borrowed from Lloyds (multi-key concurrent) ----
# Fills `lloyds_customer` ('True'/'False'/'error' strings) ONLY for rows still
# missing a value. Checks EVERY charge (any status, incl. satisfied).
# Matching covers the whole Lloyds Banking Group but NOT "Lloyd's" of London;
# trim to just r"\blloyds\b" for Lloyds Bank only.
LLOYDS_PATTERNS = [
    r"\blloyds\b",                       # Lloyds Bank, Lloyds TSB, Lloyds Commercial Finance, Lloyds Dev. Capital
    r"bank of scotland",                 # Bank of Scotland plc (Halifax's legal entity too)
    r"\bhbos\b",                         # HBOS plc
    r"agricultural mortgage corp",       # AMC — Lloyds' farm/agri lender
    r"\bhalifax\b",                      # Halifax (division of Bank of Scotland)
    r"birmingham midshires",             # BM — mortgages
    r"cheltenham (?:& |and )gloucester", # C&G — mortgages
    r"bank of wales",                    # historic BoS brand
    r"black horse",                      # Lloyds motor/asset finance
    r"scottish widows",                  # pensions/insurance
    r"\bmbna\b",                         # credit cards (Lloyds acquired 2017)
]
LLOYDS_RE = re.compile("|".join(LLOYDS_PATTERNS), re.I)

SME_OUT = Path("company_data") / "company_sme_with_acctype" / "com_names_sme.csv"
sme = pd.read_csv(SME_OUT, dtype=str)
if "lloyds_customer" not in sme.columns:
    sme["lloyds_customer"] = pd.NA


def ever_borrowed_from_lloyds(company_num):
    """'True'/'False' if any charge is held by a Lloyds entity (pages all charges);
       'error' on failure. Returns strings for a consistent dtype."""
    cnum = str(company_num).strip()
    start = 0
    while True:
        r = ch_get(f"/company/{cnum}/charges",
                   params={"items_per_page": 100, "start_index": start})
        if r is None:            return "error"
        if r.status_code == 404: return "False"   # no charges register -> never borrowed
        if r.status_code != 200: return "error"
        data = r.json()
        items = data.get("items", [])
        for ch in items:
            for p in ch.get("persons_entitled", []):
                if LLOYDS_RE.search(p.get("name", "")):
                    return "True"
        start += len(items)
        if not items or start >= data.get("total_count", 0):
            break
    return "False"


todo = sme.index[sme["lloyds_customer"].isna()]
parallel_fill(sme, todo, ever_borrowed_from_lloyds, "lloyds_customer", SME_OUT, desc="lloyds_customer")
sme["lloyds_customer"].value_counts(dropna=False)


## Add most-recent PSC kind

For each SME company, pull its Persons with Significant Control and record the `kind` of the **most recent** PSC (latest `notified_on`). Adds a `recent_psc_kind` column to `com_names_sme.csv`. Resumable — only fills rows still missing a value.

Kinds: `individual-` / `corporate-entity-` / `legal-person-` / `super-secure-person-with-significant-control`; sentinels `no_psc`, `not_found`, `error`.

In [ ]:
# --- Add `recent_psc_kind`: kind of the most recent PSC (multi-key concurrent) ---
# Fills only rows still missing it. "Most recent" = latest notified_on.
# Sentinels: 'no_psc', 'not_found', 'error'.
SME_OUT = Path("company_data") / "company_sme_with_acctype" / "com_names_sme.csv"
sme = pd.read_csv(SME_OUT, dtype=str)
if "recent_psc_kind" not in sme.columns:
    sme["recent_psc_kind"] = pd.NA


def most_recent_psc_kind(company_num):
    """Kind of the PSC with the latest notified_on, or a status sentinel."""
    r = ch_get(f"/company/{str(company_num).strip()}/persons-with-significant-control",
               params={"items_per_page": 100})
    if r is None:            return "error"
    if r.status_code == 404: return "no_psc"        # no PSC register
    if r.status_code != 200: return "error"
    items = r.json().get("items", [])
    if not items:            return "no_psc"
    latest = max(items, key=lambda it: it.get("notified_on") or "")
    return latest.get("kind") or "unknown"


todo = sme.index[sme["recent_psc_kind"].isna()]
parallel_fill(sme, todo, most_recent_psc_kind, "recent_psc_kind", SME_OUT, desc="recent_psc_kind")
sme["recent_psc_kind"].value_counts(dropna=False)


## Add accounts overdue + most-recent charge status/date

For each SME company, in one pass: pull the profile for `accounts.overdue` and the charges for the **most recent** charge's `status` and `created_on`. Adds `accounts_overdue`, `recent_charge_status`, `recent_charge_created_on` to `com_names_sme.csv`. Resumable (2 API calls per company). Sentinels: `no_charges`, `not_found`, `error`, `unknown`.

In [23]:
# --- Combined: accounts_overdue + most-recent charge status/date (multi-key) -----
# Fills three columns per company from two endpoints (2 API calls each):
#   accounts_overdue           <- profile  accounts.overdue
#   recent_charge_status       <- charges  (most recent charge, latest created_on)
#   recent_charge_created_on   <- charges
# Resumable: only processes rows still missing ANY of the three.
SME_OUT = Path("company_data") / "company_sme_with_acctype" / "com_names_sme.csv"
sme = pd.read_csv(SME_OUT, dtype=str)
COLS = ["accounts_overdue", "recent_charge_status", "recent_charge_created_on"]
for col in COLS:
    if col not in sme.columns:
        sme[col] = pd.NA


def profile_and_charge_info(company_num):
    """Return (accounts_overdue, recent_charge_status, recent_charge_created_on)."""
    cnum = str(company_num).strip()

    # profile -> accounts.overdue
    rp = ch_get(f"/company/{cnum}")
    if rp is None:              overdue = "error"
    elif rp.status_code == 404: overdue = "not_found"
    elif rp.status_code != 200: overdue = "error"
    else:
        ov = rp.json().get("accounts", {}).get("overdue")
        overdue = "unknown" if ov is None else str(ov)

    # charges -> most recent charge's status + created_on
    status = created = None
    start, charges = 0, []
    while True:
        rc = ch_get(f"/company/{cnum}/charges",
                    params={"items_per_page": 100, "start_index": start})
        if rc is None:            status = created = "error";      break
        if rc.status_code == 404: status = created = "no_charges"; break
        if rc.status_code != 200: status = created = "error";      break
        data = rc.json()
        items = data.get("items", [])
        charges.extend(items)
        start += len(items)
        if not items or start >= data.get("total_count", 0):
            break
    if status is None:                       # paged through with no error
        if charges:
            latest = max(charges, key=lambda c: c.get("created_on") or "")
            status = latest.get("status") or "unknown"
            created = latest.get("created_on") or "unknown"
        else:
            status = created = "no_charges"

    return overdue, status, created


todo = sme.index[sme[COLS].isna().any(axis=1)]
parallel_fill(sme, todo, profile_and_charge_info, COLS, SME_OUT, desc="overdue+charge")
sme[COLS].apply(lambda c: c.value_counts(dropna=False))


13544 to fetch (of 13544) — 4 keys x 8 workers
  400/13544 | 3.30/s | ETA ~66 min | checkpoint saved
  800/13544 | 3.30/s | ETA ~64 min | checkpoint saved
  1200/13544 | 3.31/s | ETA ~62 min | checkpoint saved
  1600/13544 | 3.31/s | ETA ~60 min | checkpoint saved
  2000/13544 | 3.31/s | ETA ~58 min | checkpoint saved
  2400/13544 | 3.31/s | ETA ~56 min | checkpoint saved
  2800/13544 | 3.31/s | ETA ~54 min | checkpoint saved
  3200/13544 | 3.31/s | ETA ~52 min | checkpoint saved
  3600/13544 | 3.31/s | ETA ~50 min | checkpoint saved
  4000/13544 | 3.31/s | ETA ~48 min | checkpoint saved
  4400/13544 | 3.31/s | ETA ~46 min | checkpoint saved
  4800/13544 | 3.31/s | ETA ~44 min | checkpoint saved
  5200/13544 | 3.31/s | ETA ~42 min | checkpoint saved
  5600/13544 | 3.31/s | ETA ~40 min | checkpoint saved
  6000/13544 | 3.31/s | ETA ~38 min | checkpoint saved
  6400/13544 | 3.31/s | ETA ~36 min | checkpoint saved
  6800/13544 | 3.31/s | ETA ~34 min | checkpoint saved
  7200/13544 | 3.31/

,accounts_overdue,recent_charge_status,recent_charge_created_on
1970-10-23,NaN,NaN,1.0
1974-04-30,NaN,NaN,1.0
1974-06-12,NaN,NaN,1.0
1975-01-13,NaN,NaN,1.0
1975-03-04,NaN,NaN,1.0
...,...,...,...
True,327.0,NaN,NaN
fully-satisfied,NaN,271.0,NaN
no_charges,NaN,11759.0,11759.0
outstanding,NaN,1513.0,NaN


## Normalise types for analysis

Load a typed view (`sme_typed`) with `lloyds_customer` as a real boolean, so filtering and `== True` work. The CSV on disk stays uniform text.

In [24]:
# --- Analysis-ready load: consistent dtypes & safe comparisons -----------------
# The CSV stores everything as text (uniform "True"/"False"/sentinels). This loads
# a typed view where the boolean flags become real nullable booleans, so you can do
# `sme_typed[sme_typed.lloyds_customer]` or `== True`. Non True/False values
# (e.g. 'error') become <NA> and are reported so nothing is hidden.
SME_OUT = Path("company_data") / "company_sme_with_acctype" / "com_names_sme.csv"
sme_typed = pd.read_csv(SME_OUT, dtype=str)

bool_map = {"True": True, "False": False}
for col in ("lloyds_customer", "accounts_overdue"):
    if col in sme_typed.columns:
        odd = sme_typed.loc[sme_typed[col].notna() & ~sme_typed[col].isin(bool_map), col]
        if len(odd):
            print(f"Non True/False {col} values -> <NA>:")
            print(odd.value_counts().to_string(), "\n")
        sme_typed[col] = sme_typed[col].map(bool_map).astype("boolean")
        print(f"{col}:")
        print(sme_typed[col].value_counts(dropna=False).to_string(), "\n")

for col in ("recent_psc_kind", "recent_charge_status"):
    if col in sme_typed.columns:
        print(f"{col}:")
        print(sme_typed[col].value_counts(dropna=False).to_string(), "\n")

# `sme_typed` is analysis-ready (e.g. sme_typed[sme_typed.lloyds_customer]).
# The CSV on disk stays canonical text.
sme_typed.head()


lloyds_customer:
lloyds_customer
False    13308
True       236 

accounts_overdue:
accounts_overdue
False    13217
True       327 

recent_psc_kind:
recent_psc_kind
individual-person-with-significant-control          9801
no_psc                                              2839
corporate-entity-person-with-significant-control     887
legal-person-person-with-significant-control          17 

recent_charge_status:
recent_charge_status
no_charges         11759
outstanding         1513
fully-satisfied      271
part-satisfied         1 



,com_num,name,sic_code,account_type,lloyds_customer,recent_psc_kind,recent_charge_status,recent_charge_created_on,accounts_overdue
0,15073164,NFLECTION ADVISORY LIMITED,70229,total-exemption-full,False,individual-person-with-significant-control,no_charges,no_charges,False
1,13522064,NFOGENIE LTD,58290,micro-entity,False,individual-person-with-significant-control,no_charges,no_charges,False
2,11006939,NNOV8 LIMITED,62090,micro-entity,False,individual-person-with-significant-control,no_charges,no_charges,False
3,SC606050,NSPIRED INVESTMENTS LTD,68209,total-exemption-full,False,individual-person-with-significant-control,outstanding,2021-09-01,False
4,11303802,"""1ST RATE"" PSYCHOLOGY SERVICES LTD",85600,micro-entity,False,individual-person-with-significant-control,no_charges,no_charges,False


## Repair com_num leading zeros

Realign `com_num` in `com_names_sme.csv` to the canonical values in `company_raw/` (fixes leading zeros dropped by Excel, e.g. `5914136` → `05914136`). Idempotent — run it any time, especially after opening/saving the CSV in a spreadsheet.

In [ ]:
# --- Repair com_num: realign leading zeros to the raw originals ------------------
# UK company numbers keep leading zeros (e.g. 05914136). Opening the CSV in Excel
# can silently drop them (05914136 -> 5914136, or float-ify to 5914136.0). This
# remaps com_num back to the canonical values in company_raw/. Idempotent:
# reports 0 changed when the file is already aligned.
RAW_DIR = Path("company_data") / "company_raw"
SME_OUT = Path("company_data") / "company_sme_with_acctype" / "com_names_sme.csv"


def _norm(x):
    """Collapse a com_num to a match key: drop a stray '.0', then leading zeros."""
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x.lstrip("0")


# canonical map: normalized form -> original com_num (from the raw files)
raw = pd.concat([pd.read_csv(f, dtype=str) for f in sorted(RAW_DIR.glob("*.csv"))],
                ignore_index=True).drop_duplicates("com_num")
canon = {_norm(c): c for c in raw["com_num"]}

sme = pd.read_csv(SME_OUT, dtype=str)
before = sme["com_num"].copy()
sme["com_num"] = sme["com_num"].map(lambda x: canon.get(_norm(x), str(x).strip()))

changed = int((before != sme["com_num"]).sum())
unmatched = sme.loc[~sme["com_num"].isin(set(raw["com_num"])), "com_num"]
sme.to_csv(SME_OUT, index=False)
print(f"Aligned com_num -> {SME_OUT}")
print(f"  rows changed: {changed}")
print(f"  not matching any raw com_num: {len(unmatched)}")
if len(unmatched):
    print("  examples:", unmatched.head(5).tolist())
